<a href="https://colab.research.google.com/github/namgiju/mlprog/blob/main/mobile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

# 구글 드라이브 연결
drive.mount('/content/drive')

# CSV 불러오기
df = pd.read_csv('/content/drive/MyDrive/mobile.csv')

# 데이터 확인
print(df.shape)
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(2000, 21)
   battery_power  blue  clock_speed  dual_sim  fc  four_g  int_memory  m_dep  \
0            842     0          2.2         0   1       0           7    0.6   
1           1021     1          0.5         1   0       1          53    0.7   
2            563     1          0.5         1   2       1          41    0.9   
3            615     1          2.5         0   0       0          10    0.8   
4           1821     1          1.2         0  13       1          44    0.6   

   mobile_wt  n_cores  ...  px_height  px_width   ram  sc_h  sc_w  talk_time  \
0        188        2  ...         20       756  2549     9     7         19   
1        136        3  ...        905      1988  2631    17     3          7   
2        145        5  ...       1263      1716  2603    11     2          9   
3        131        6  ...       1216      1786  2769    16

가격 등급 분류

In [4]:
# 입력값과 정답 분리
X = df.drop('price_range', axis=1)
y = df['price_range']

# 학습 / 테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y
)

# 표준화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 딥러닝 모델
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(20,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 학습
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

# 평가
loss, accuracy = model.evaluate(X_test, y_test)
print("정확도:", accuracy)

# 예측
pred = model.predict(X_test)
print("예측 등급:", np.argmax(pred, axis=1)[:10])

Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.3633 - loss: 1.3451 - val_accuracy: 0.4094 - val_loss: 1.2818
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5336 - loss: 1.1371 - val_accuracy: 0.5219 - val_loss: 1.0894
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6438 - loss: 0.9173 - val_accuracy: 0.6469 - val_loss: 0.8836
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7305 - loss: 0.7268 - val_accuracy: 0.7312 - val_loss: 0.7076
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8117 - loss: 0.5873 - val_accuracy: 0.7812 - val_loss: 0.5899
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8703 - loss: 0.4819 - val_accuracy: 0.8531 - val_loss: 0.4947
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9031 - loss: 0.4007 - val_accuracy: 0.8625 - val_loss: 0.4298
Epoch 8/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9164 - loss: 0.3370 - val_accuracy: 0.8750 - 

배터리 용량 예측

In [5]:
# 가격 등급 열 삭제
df_reg = df.drop('price_range', axis=1)

# 입력값과 정답 분리
X = df_reg.drop('battery_power', axis=1)
y = df_reg['battery_power']

# 학습 / 테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 표준화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 딥러닝 모델
model_reg = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(19,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model_reg.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# 학습
model_reg.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

# 예측
pred = model_reg.predict(X_test).flatten()

# 평가
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("MAE:", mae)
print("RMSE:", rmse)

# 실제값과 예측값 비교
result = pd.DataFrame({
    '실제 배터리 용량': y_test.values,
    '예측 배터리 용량': pred
})

print(result.head(10))

Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1709756.2500 - mae: 1232.5085 - val_loss: 1813708.0000 - val_mae: 1267.6716
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1700545.6250 - mae: 1228.8025 - val_loss: 1800530.7500 - val_mae: 1262.4277
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1681839.2500 - mae: 1221.1340 - val_loss: 1774169.3750 - val_mae: 1251.8826
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1646614.2500 - mae: 1206.6891 - val_loss: 1727706.7500 - val_mae: 1233.0745
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1587642.2500 - mae: 1182.0734 - val_loss: 1653977.2500 - val_mae: 1202.6235
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1498294.1250 - mae: 1144.0006 - val_loss: 1547278.3750 - val_mae: 1157.0460
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1374556.2500 - mae: 1088.3068 - val_loss: 1405875.2500 - val_mae: 1093.6150
Epoch 8/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - lo